# Fine-tune Qwen3.5-35B-A3B on Graphiti extraction (memory set)

Spec: `specs/finetune-35b-pilot.md`. Data: private HF dataset `aayush7511/graphiti-memory-synth` (config `examples`).

**Before running**
1. Runtime → Change runtime type → an **80 GB GPU** (A100 80GB or H100). bf16 LoRA needs ~74 GB; 4-bit QLoRA is not recommended for this MoE.
2. Secrets (key icon, left sidebar) → add `HF_TOKEN` (a Hugging Face token with write access) and enable notebook access.

Output: LoRA adapter pushed to the private model repo `aayush7511/qwen3.5-35b-a3b-graphiti-lora`.

Install and model-loading cells are adapted from Unsloth's `Qwen3_5_MoE.ipynb` (LGPL-3.0, https://github.com/unslothai/notebooks).

## Install (unchanged from Unsloth)

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
%%capture
# Unsloth bundles the gated delta net kernels, so flash-linear-attention does
# not need to be installed here.
import json, platform, sys, torch
from urllib.request import urlopen

py   = f"cp{sys.version_info.major}{sys.version_info.minor}"
tv   = ".".join(torch.__version__.split("+")[0].split(".")[:2])
tv   = {"2.11": "2.10", "2.12": "2.10"}.get(tv, tv)
assert (cu := (torch.version.cuda or "").split(".")[0]), "CUDA-enabled PyTorch required."
abi  = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
plat = "linux_x86_64" if platform.machine().lower() in ("x86_64", "amd64") else "linux_aarch64"
whl  = f"{py}-{py}-{plat}.whl"

def find(repo, tag, match):
    api = f"https://api.github.com/repos/{repo}/releases/tags/{tag}"
    return next((a["browser_download_url"] for a in json.load(urlopen(api))["assets"] if match(a["name"])), None)

cc1d = find("Dao-AILab/causal-conv1d", "v1.6.1.post4",
            lambda n: n.endswith(whl) and f"+cu{cu}torch{tv}" in n and f"cxx11abi{abi}" in n)
if not cc1d:
    print(f"No prebuilt causal-conv1d wheel for torch {torch.__version__}/cu{cu}/{py}/abi{abi}, skipping it.")
wheels = " ".join(f'"{w}"' for w in (cc1d,) if w)
!test {len(wheels)} -eq 0 || uv pip install -qqq --no-deps $wheels
!pip uninstall -y sentence-transformers torchcodec  # torchcodec import broken on Colab

## GPU check and Hugging Face token

In [ ]:
import torch
from google.colab import userdata
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.0f} GiB")
assert gpu.total_memory > 70 * 2**30, "Needs an 80 GB GPU (A100 80GB / H100): Runtime -> Change runtime type"
HF_TOKEN = userdata.get("HF_TOKEN")  # Secrets panel; never paste the token into a cell
ADAPTER_REPO = "aayush7511/qwen3.5-35b-a3b-graphiti-lora"

## Model: bf16 LoRA, 6,144-token sequences (our examples: median 3,389, max 5,695 tokens)

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 6144
lora_rank = 16

model, processor = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3.5-35B-A3B",  # ~70 GB download, takes a while
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    fast_inference = False,  # not supported for MoE yet
)
tokenizer = processor.tokenizer

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "gate_up_proj",  # LoRA on MoE layers too
    ],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    bias = "none",
)

## Data: Graphiti prompt → gold JSON, thinking off (Graphiti calls the model without it)

In [ ]:
from datasets import load_dataset

data = load_dataset("aayush7511/graphiti-memory-synth", "examples", token = HF_TOKEN)

def to_text(batch):
    texts = []
    for messages, target in zip(batch["messages"], batch["target"]):
        convo = list(messages) + [{"role": "assistant", "content": target}]
        texts.append(tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False,
                                                   enable_thinking = False))
    return {"text": texts}

train = data["train"].map(to_text, batched = True)
val = data["validation"].map(to_text, batched = True)
lengths = sorted(len(tokenizer(t).input_ids) for t in train["text"])
print(len(train), "train /", len(val), "validation; tokens median", lengths[len(lengths)//2], "max", lengths[-1])
assert lengths[-1] <= max_seq_length, "an example is longer than max_seq_length"
print(train[0]["text"][-1200:])

## Trainer: 2 epochs, loss on the answer only

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    eval_dataset = val,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        warmup_steps = 10,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        save_strategy = "no",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
# Only the answer should survive the mask:
print(tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]])
      .replace(tokenizer.pad_token, "")[-600:])

In [ ]:
# Compilation takes a few minutes at the start.
stats = trainer.train()
print(stats)

## Check: answer one held-out Graphiti prompt

In [ ]:
import json
FastLanguageModel.for_inference(model)
ex = data["validation"][1]  # an extract_edges example
prompt = tokenizer.apply_chat_template(list(ex["messages"]), tokenize = False, add_generation_prompt = True,
                                       enable_thinking = False)
out = model.generate(**tokenizer(prompt, return_tensors = "pt").to("cuda"), max_new_tokens = 1500,
                     temperature = 0.0, do_sample = False)
answer = tokenizer.decode(out[0][len(tokenizer(prompt).input_ids):], skip_special_tokens = True).strip()
print(ex["step"], "\n--- model ---\n", answer[:1500], "\n--- gold ---\n", ex["target"][:1500])
json.loads(answer)  # must be valid JSON

## Save: push the LoRA adapter to a private HF repo

In [ ]:
from huggingface_hub import create_repo
create_repo(ADAPTER_REPO, private = True, exist_ok = True, token = HF_TOKEN)
model.push_to_hub(ADAPTER_REPO, token = HF_TOKEN, private = True)
tokenizer.push_to_hub(ADAPTER_REPO, token = HF_TOKEN, private = True)
print("pushed", ADAPTER_REPO)